Fase 1: Exploración de la base de datos y construcción del Dataset

Integrantes:

Alison Diaz

Lucas Dominguez

Lautaro Pérez

Dylan Cabral

Ayelén Ortega

SuperTienda no tiene un dataset consolidado. Por eso, se explora la relación de la tablas en la base de datos para construir un dataset limpio, desenchando datos irrelevantes.

In [1]:
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%sql sqlite:///SuperTienda.db

In [2]:
%%sql
SELECT name
FROM sqlite_master
WHERE type='table';

 * sqlite:///SuperTienda.db
Done.


name
Clientes
Productos
Geografia
Cliente_Ubicacion
Campanias
Pedidos
Detalle_Pedido
Soporte


La base de datos presenta 8 tablas

Elegimos trabajar las tablas Clientes, Pedidos, Detalle_Pedido, Productos, Cliente_Ubicacion y Geografia porque contienen la información necesaria para relacionar clientes, ventas, productos y ubicación geográfica. Las tablas Campanias y Soporte fueron descartadas porque no aportan información relevante al análisis


In [3]:
%%sql
SELECT *
FROM Clientes
LIMIT 5;

 * sqlite:///SuperTienda.db
Done.


ID_Cliente,Nombre_Cliente,Segmento,Fecha_Registro,Canal_Preferido,Puntaje_Fidelidad,Codigo_Interno
C0001,Fernanda Díaz,Consumidor,2023-02-18,Web,55,CLI-6635-Z
C0002,Valentina Muñoz,Home Office,2023-07-30,Web,58,CLI-2291-Z
C0003,Diego Castillo,Corporativo,2024-03-28,App,100,CLI-2139-X
C0004,Nicolás Silva,Corporativo,2021-06-13,App,22,CLI-7227-Y
C0005,Daniela Rodríguez,Consumidor,2023-01-29,Teléfono,36,CLI-5374-Z


In [4]:
%%sql
SELECT *
FROM Productos
LIMIT 5;

 * sqlite:///SuperTienda.db
Done.


ID_Producto,Nombre_Producto,Categoria,Subcategoria,Precio_Lista,Costo_Unitario,Marca,Activo,Codigo_Barra
P0001,Accesorios Flex 1,Tecnología,Accesorios,445.84,210.5,Alerce,1,7154379937124
P0002,Accesorios Smart 2,Tecnología,Accesorios,10.06,5.92,Vector,1,5756924593752
P0003,Accesorios Plus 3,Tecnología,Accesorios,868.26,685.1,Altura,1,3564651939828
P0004,Accesorios Eco 4,Tecnología,Accesorios,28.12,19.81,Orion,1,4167243426617
P0005,Accesorios Flex 5,Tecnología,Accesorios,891.67,550.81,Vector,1,5891135290965


In [5]:
%%sql
SELECT *
FROM Geografia
LIMIT 5;

 * sqlite:///SuperTienda.db
Done.


ID_Ubicacion,Pais,Region,Estado,Ciudad,Codigo_Postal,Zona_Logistica,Latitud_Ref
UB001,Chile,Norte,Antofagasta,Antofagasta,1240000,A,-52.1246
UB002,Chile,Norte,Coquimbo,La Serena,1700000,C,-44.4288
UB003,Chile,Norte,Tarapacá,Iquique,1100000,B,-27.2235
UB004,Chile,Centro,Metropolitana,Santiago,8320000,A,-32.3328
UB005,Chile,Centro,Valparaíso,Valparaíso,2340000,A,-51.9571


In [6]:
%%sql
SELECT *
FROM Cliente_Ubicacion
LIMIT 5;

 * sqlite:///SuperTienda.db
Done.


ID_Cliente,ID_Ubicacion,Tipo_Direccion,Fecha_Actualizacion
C0001,UB003,Principal,2024-01-08
C0002,UB012,Principal,2024-06-21
C0003,UB013,Principal,2024-05-31
C0004,UB010,Principal,2024-12-11
C0005,UB013,Principal,2024-04-08


In [7]:
%%sql
SELECT *
FROM Pedidos
LIMIT 5;

 * sqlite:///SuperTienda.db
Done.


ID_Pedido,ID_Cliente,Fecha_Pedido,Fecha_Envio,Modo_Envio,ID_Campania,Estado_Pedido,Observacion_Interna
ORD-2023-00001,C0174,2025-03-31,2025-04-04,Estándar,None,Completado,Cliente frecuente
ORD-2023-00002,C0254,2024-03-25,2024-03-25,Segunda Clase,MKT005,Completado,Sin observaciones
ORD-2023-00003,C0203,2024-02-20,2024-02-24,Estándar,MKT001,Completado,Revisar descuento aplicado
ORD-2023-00004,C0258,2023-10-29,2023-10-31,Primera Clase,None,Enviado,Sin observaciones
ORD-2023-00005,C0118,2025-03-30,2025-04-01,Estándar,MKT006,Completado,Verificar dirección


In [8]:
%%sql
SELECT *
FROM Detalle_Pedido
LIMIT 5;

 * sqlite:///SuperTienda.db
Done.


ID_Detalle,ID_Pedido,ID_Producto,Cantidad,Descuento,Ventas,Ganancia,Prioridad
1,ORD-2023-00001,P0082,1,0.0,590.86,322.42,Crítica
2,ORD-2023-00001,P0119,7,0.0,4341.68,1907.85,Media
3,ORD-2023-00001,P0048,2,0.0,1250.86,281.32,Crítica
4,ORD-2023-00001,P0079,5,0.0,244.35,64.85,Media
5,ORD-2023-00002,P0087,1,0.15,629.31,115.33,Baja


**Fase 2: Extracción de datos (SQL y Pandas)**

Para la construccion del Dataset se tomó el enfoque de Rentabilidad y Ventas.

Para relacionar las tablas que utilizaremos, haremos uso de JOIN.




In [9]:
%%sql resultado <<
SELECT
    c.ID_Cliente,
    c.Nombre_Cliente,
    c.Segmento,

    g.Pais,
    g.Region,
    g.Estado,
    g.Ciudad,

    p.ID_Pedido,
    p.Fecha_Pedido,

    pr.ID_Producto,
    pr.Nombre_Producto,
    pr.Categoria,
    pr.Subcategoria,

    dp.Cantidad,
    dp.Descuento,
    dp.Ventas,
    dp.Ganancia

FROM Clientes c

JOIN Pedidos p ON c.ID_Cliente = p.ID_Cliente
JOIN Detalle_Pedido dp ON p.ID_Pedido = dp.ID_Pedido
JOIN Productos pr ON dp.ID_Producto = pr.ID_Producto
JOIN Cliente_Ubicacion cu ON c.ID_Cliente = cu.ID_Cliente
JOIN Geografia g ON cu.ID_Ubicacion = g.ID_Ubicacion;


 * sqlite:///SuperTienda.db
Done.
Returning data to local variable resultado


Convertimos la consulta SQL en un dataframe

In [10]:
import pandas as pd
df = resultado.DataFrame()
df.head()

,ID_Cliente,Nombre_Cliente,Segmento,Pais,Region,Estado,Ciudad,ID_Pedido,Fecha_Pedido,ID_Producto,Nombre_Producto,Categoria,Subcategoria,Cantidad,Descuento,Ventas,Ganancia
0,C0174,Nicolás Martínez,Corporativo,Chile,Sur,Los Lagos,Puerto Montt,ORD-2023-00001,2025-03-31,P0082,Papelería Eco 2,Oficina,Papelería,1,0.00,590.86,322.42
1,C0174,Nicolás Martínez,Corporativo,Chile,Sur,Los Lagos,Puerto Montt,ORD-2023-00001,2025-03-31,P0119,Sobres Plus 9,Oficina,Sobres,7,0.00,4341.68,1907.85
2,C0174,Nicolás Martínez,Corporativo,Chile,Sur,Los Lagos,Puerto Montt,ORD-2023-00001,2025-03-31,P0048,Sillas Plus 8,Muebles,Sillas,2,0.00,1250.86,281.32
3,C0174,Nicolás Martínez,Corporativo,Chile,Sur,Los Lagos,Puerto Montt,ORD-2023-00001,2025-03-31,P0079,Almacenamiento Nova 9,Muebles,Almacenamiento,5,0.00,244.35,64.85
4,C0254,Isidora Rojas,Consumidor,Chile,Sur,Biobío,Concepción,ORD-2023-00002,2024-03-25,P0087,Papelería Flex 7,Oficina,Papelería,1,0.15,629.31,115.33


Convertimos las fechas en numeros reales para que sea más fácil de leer o acceder a fechas como mes, o año. Si no, se leerian como strings y dificultaría consultas posteriores.

In [17]:
df.describe()
df["Fecha_Pedido"] = pd.to_datetime(df["Fecha_Pedido"])

**Consulta 1**: Qué categorías generan mayores ventas

In [19]:
#Haremos el analisis de la primera consulta que hicimos
ventas_categoria = df.groupby("Categoria")["Ventas"].sum()
ventas_categoria
#Agrupamos las ventas por categoria para saber cual vendió más y generó más ganancias

,Ventas
Categoria,
Muebles,1631940.33
Oficina,1811897.47
Tecnología,2301408.92


**Consulta 2**: Qué regiones producen más ganancias

In [21]:
#Haremos el analisis de la segunda consulta
ganancia_region = df.groupby("Region")["Ganancia"].sum()
ganancia_region
#Qué región es más rentable para la empresa

,Ganancia
Region,
Austral,288112.92
Centro,572075.72
Norte,387572.21
Sur,554926.92


**Consulta 3**: El segmento de clientes, ¿Influye en la rentabilidad del negocio?

In [20]:
#Haremos el analisis de la tercera consulta
ganancia_segmento = df.groupby("Segmento")["Ganancia"].sum()
ganancia_segmento
#Hay un segmento que genera más ganancia o compra más


,Ganancia
Segmento,
Consumidor,594072.79
Corporativo,705527.47
Home Office,503087.51


**Fase 3: Limpieza y preparación**

Se inicia el proceso de preparación del dataset analítico.



En esta etapa se verificó la calidad del dataset para asegurar que la información fuera consistente y adecuada para el análisis. Se revisaron los tipos de datos, la existencia de valores nulos y registros duplicados

In [24]:
df.info()
df.isnull().sum()
df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3026 entries, 0 to 3025
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   ID_Cliente       3026 non-null   object        
 1   Nombre_Cliente   3026 non-null   object        
 2   Segmento         3026 non-null   object        
 3   Pais             3026 non-null   object        
 4   Region           3026 non-null   object        
 5   Estado           3026 non-null   object        
 6   Ciudad           3026 non-null   object        
 7   ID_Pedido        3026 non-null   object        
 8   Fecha_Pedido     3026 non-null   datetime64[ns]
 9   ID_Producto      3026 non-null   object        
 10  Nombre_Producto  3026 non-null   object        
 11  Categoria        3026 non-null   object        
 12  Subcategoria     3026 non-null   object        
 13  Cantidad         3026 non-null   int64         
 14  Descuento        3026 non-null   float64

np.int64(0)

Variable calculada: calculamos el porcentaje de ganancia de cada venta

De esta manera podemos medir la retabilidad de cada venta, de esta manera es más fácil la comparacion entre productos, categoía y segmentos

In [23]:
df["Margen_Ganancia"] = (df["Ganancia"] / df["Ventas"]) * 100
#Verificación
df[["Ventas", "Ganancia", "Margen_Ganancia"]].head()

,Ventas,Ganancia,Margen_Ganancia
0,590.86,322.42,54.567918
1,4341.68,1907.85,43.942667
2,1250.86,281.32,22.490127
3,244.35,64.85,26.539799
4,629.31,115.33,18.326421


Fase 4: Exportación del dataset

Pasamos a limpio en un archivo csv y descargamos el archivo en un google colab

In [26]:
df.to_csv("DataSet_Limpio.csv", index=False)
from google.colab import files
files.download("DataSet_Limpio.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Resumen Analítico**

Fase 1:

Tomamos la base de datos de SuperTienda, y analizamos sus tablas en general. Conocemos las relaciones entre ellas y desechamos los datos irrelevantes.

Fase 2:

**Enfoque**: Rentabilidad y Ventas

**Objetivo del Proyecto**:
Analizar la rentabilidad de la empresa SuperTienda para identificar *qué categorías generan mayores ventas*, *qué regiones producen más ganancias* y si el
*segmento de clientes influye en la rentabilidad* del negocio. Para eso, se tuvieron en cuenta las tablas Clientes, Geografía, Pedidos, Productos y Detalle_Pedido
De esta manera se hace un Dataset completo y donde, en el futuro, servirá para diferentes análisis sin tener que recurrir a la base de datos.

Fase 3:

Hacemos una preparación al dataset para que sea consistente y fácil de interpretar para los analisis.

Fase 4:

Una vez limpio el Dataset, se convierte a formato CSV para su exportación

Este Dataset también facilita el analisis de rentabilidad de la empresa con dashboards (Ej: Power BI) y observar la evolución de las ganancias a lo largo del tiempo, comparar el rendimiento por categoría de producto, identificar las regiones o ciudades más rentables y evaluar el comportamiento de los distintos segmentos de clientes. De esta manera, el Dataset se vuelve una base para obtener información esencial para la toma de decisiones.